<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/60_pack_and_publish_to_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==== Copy app artefacts from Drive into Git repo and push ====
from getpass import getpass
import os, shutil, subprocess
from pathlib import Path

# --- CONFIG ---
GITHUB_USER = "brendonhuynhbp-hub"
REPO_NAME   = "gt-markets"
BRANCH      = "main"

REPO_DIR    = f"/content/{REPO_NAME}"
SRC_ARTE    = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")
DST_ARTE    = Path(REPO_DIR) / "AppDemo" / "artefacts"

# --- Mount Drive just in case ---
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --- Prompt for GitHub token securely ---
print("Enter a GitHub Personal Access Token (with repo scope). It will NOT be printed.")
TOKEN = getpass("GitHub Token: ").strip()
assert TOKEN, "No token provided."

# --- Prepare GIT_ASKPASS helper (no token in logs) ---
ASKPASS = "/content/git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write(f"""#!/bin/sh
case "$1" in
  *Username*) echo "{GITHUB_USER}";;
  *Password*) echo "{TOKEN}";;
  *) echo "{TOKEN}";;
esac
""")
os.chmod(ASKPASS, 0o700)
env = os.environ.copy()
env["GIT_ASKPASS"] = ASKPASS
env["GIT_TERMINAL_PROMPT"] = "0"

# --- Clone repo clean ---
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone",f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git", REPO_DIR],
               check=True, env=env)
subprocess.run(["git","checkout",BRANCH], cwd=REPO_DIR, check=True, env=env)

# --- Replace artefacts folder ---
if DST_ARTE.exists():
    shutil.rmtree(DST_ARTE)
shutil.copytree(SRC_ARTE, DST_ARTE)

print("✅ Copied artefacts into repo:", DST_ARTE)

# --- Commit & push ---
subprocess.run(["git","config","user.email","you@example.com"], cwd=REPO_DIR, check=True)
subprocess.run(["git","config","user.name","Your Name"], cwd=REPO_DIR, check=True)

subprocess.run(["git","add","-A"], cwd=REPO_DIR, check=True)
subprocess.run(["git","commit","-m","Update artefacts from main notebook"], cwd=REPO_DIR, check=False)

try:
    subprocess.run(["git","push","origin",BRANCH], cwd=REPO_DIR, check=True, env=env)
    print("🚀 Pushed artefacts to GitHub. Streamlit Cloud will pick them up on next deploy.")
except subprocess.CalledProcessError as e:
    print("Push failed:", e.stderr or e.stdout)


Mounted at /content/drive
Enter a GitHub Personal Access Token (with repo scope). It will NOT be printed.
GitHub Token: ··········
✅ Copied artefacts into repo: /content/gt-markets/AppDemo/artefacts
🚀 Pushed artefacts to GitHub. Streamlit Cloud will pick them up on next deploy.
